In [5]:
import pandas as pd
import numpy as np

# 1. Load the raw Kaggle datasets
# Ensure these files are uploaded to your SageMaker notebook environment
df1 = pd.read_csv('1st.csv')
df2 = pd.read_csv('2nd.csv')
df3 = pd.read_csv('3rd.csv')

# 2. Map file-specific columns to the target Nafas schema
map_df1 = {
    'Patient_ID': 'patient_id',
    'Gender': 'sex',
    'Age': 'age',
    'BMI': 'BMI',
    'Air_Pollution_Level': 'IAQI',
    'Physical_Activity_Level': 'activity_level',
    'Peak_Expiratory_Flow': 'latest_pefr',
    'Has_Asthma': 'attack_predicted'
}
map_df2 = {
    'Gender': 'sex',
    'Age': 'age',
    'BMI': 'BMI',
    'Air_Pollution_Index': 'IAQV',
    'Dust_Exposure_Level': 'VOC_index',
    'Oxygen_Saturation': 'SpO2_percent',
    'Asthma': 'attack_predicted'
}
map_df3 = {
    'PatientID': 'patient_id',
    'Gender': 'sex',
    'Age': 'age',
    'BMI': 'BMI',
    'PollutionExposure': 'IAQV',
    'DustExposure': 'VOC_index',
    'PhysicalActivity': 'activity_value',
    'Diagnosis': 'attack_predicted'
}

# Apply renaming mappings
df1 = df1.rename(columns=map_df1)
df2 = df2.rename(columns=map_df2)
df3 = df3.rename(columns=map_df3)

# Extract the new target column names from your dictionaries
target_cols1 = list(map_df1.values())
target_cols2 = list(map_df2.values())
target_cols3 = list(map_df3.values())

# Filter each DataFrame to keep only those columns
df1 = df1[target_cols1]
df2 = df2[target_cols2]
df3 = df3[target_cols3]


# Filter each DataFrame for ages 5 through 12 (inclusive)
df1 = df1[df1['age'].between(5, 12)]
df2 = df2[df2['age'].between(5, 12)]
df3 = df3[df3['age'].between(5, 12)]

# Merge the three filtered dataframes
df_merged = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

df_merged['temp'] = np.nan
df_merged['humidity'] = np.nan
df_merged['heart_rate'] = np.nan

print("Total patients (Ages 5-12):", len(df_merged))
print("Total attacks:", int(df_merged['attack_predicted'].sum()))
print("\nMissing Values Table:")
print(df_merged.isna().sum())

Total patients (Ages 5-12): 1252
Total attacks: 284

Missing Values Table:
patient_id           172
sex                    0
age                    0
BMI                    0
IAQI                 397
activity_level       397
latest_pefr          397
attack_predicted       0
IAQV                 855
VOC_index            855
SpO2_percent        1080
activity_value      1027
temp                1252
humidity            1252
heart_rate          1252
dtype: int64


In [6]:
# Drop the patient_id column
df_merged = df_merged.drop(columns=['patient_id'])

# Verify the column is removed by checking the remaining columns
print(df_merged.columns.tolist())

['sex', 'age', 'BMI', 'IAQI', 'activity_level', 'latest_pefr', 'attack_predicted', 'IAQV', 'VOC_index', 'SpO2_percent', 'activity_value', 'temp', 'humidity', 'heart_rate']


In [8]:
import numpy as np
import pandas as pd

def process_air_quality(row):
    # 1. Handle IAQV: Round existing decimals to whole numbers, or impute if missing
    if pd.notna(row['IAQV']):
        iaqv = round(float(row['IAQV']))
    else:
        # Impute missing values based on attack_predicted
        if row['attack_predicted'] == 0:
            iaqv = np.random.randint(0, 101)    # 0 to 100
        elif row['attack_predicted'] == 1:
            iaqv = np.random.randint(201, 301)  # > 200
        else:
            # Fallback if attack_predicted is somehow NaN
            iaqv = np.random.randint(0, 301) 

    # 2. Overwrite/Assign IAQI strictly based on the final IAQV bracket
    if iaqv <= 100:
        iaqi = 'Excellent'
    elif 101 <= iaqv <= 200:
        iaqi = 'Moderate'
    else:
        iaqi = 'Polluted'
        
    row['IAQV'] = iaqv
    row['IAQI'] = iaqi
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_air_quality, axis=1)

# Convert the IAQV column to strict integer format to permanently remove decimal formatting
df_merged['IAQV'] = df_merged['IAQV'].astype(int)

# Verify the changes
print(df_merged[['attack_predicted', 'IAQV', 'IAQI']].head(10))

   attack_predicted  IAQV       IAQI
0                 0    26  Excellent
1                 0    65  Excellent
2                 0    80  Excellent
3                 1   222   Polluted
4                 0    85  Excellent
5                 0    53  Excellent
6                 0    54  Excellent
7                 0     2  Excellent
8                 0    67  Excellent
9                 0    92  Excellent


In [10]:
import numpy as np
import pandas as pd

def process_pefr(row):
    pefr = row['latest_pefr']
    
    # 1. Impute missing values based on attack_predicted
    if pd.isna(pefr):
        if row['attack_predicted'] == 1:
            # Assign a random float between 100 and 199 (<200)
            pefr = np.random.uniform(100, 199.9)
        elif row['attack_predicted'] == 0:
            # Assign a random float between 320 and 400
            pefr = np.random.uniform(320, 400)
        else:
            # Fallback if attack_predicted is also NaN
            pefr = np.random.uniform(200, 320)
    else:
        pefr = float(pefr)

    # 2. Assign pefr_zone strictly based on the numeric value
    if pefr < 200:
        zone = 'r'
    elif 200 <= pefr < 320:
        zone = 'y'
    else:
        # Captures 320-400 and any natural outliers above 400
        zone = 'g'
        
    row['latest_pefr'] = round(pefr, 1)  # Keeps decimal formatting consistent
    row['pefr_zone'] = zone
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_pefr, axis=1)

# Verify the changes
print(df_merged[['attack_predicted', 'latest_pefr', 'pefr_zone']].head(10))

   attack_predicted  latest_pefr pefr_zone
0                 0        499.9         g
1                 0        252.4         y
2                 0        600.0         g
3                 1        319.4         y
4                 0        330.7         g
5                 0        446.2         g
6                 0        383.5         g
7                 0        258.3         y
8                 0        273.9         y
9                 0        308.9         y


In [12]:
import numpy as np
import pandas as pd

def process_spo2(row):
    spo2 = row['SpO2_percent']
    
    # 1. Impute missing values based on attack_predicted
    if pd.isna(spo2):
        if row['attack_predicted'] == 0:
            # Normal: Random float between 97.0 and 100.0
            spo2 = np.random.uniform(97.0, 100.0)
        elif row['attack_predicted'] == 1:
            # Low: Random float between 90.0 and 96.9
            spo2 = np.random.uniform(90.0, 96.9)
        else:
            # Fallback if attack_predicted is also NaN
            spo2 = np.random.uniform(95.0, 100.0) 
    else:
        spo2 = float(spo2)

    # 2. Assign SpO2_category strictly based on the numeric value
    if spo2 >= 97:
        category = 'normal'
    else:
        category = 'low'
        
    row['SpO2_percent'] = round(spo2, 1)
    row['SpO2_category'] = category
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_spo2, axis=1)

# Verify the changes
print(df_merged[['attack_predicted', 'SpO2_percent', 'SpO2_category']].head(10))

   attack_predicted  SpO2_percent SpO2_category
0                 0          98.3        normal
1                 0          97.5        normal
2                 0          99.5        normal
3                 1          92.0           low
4                 0          99.0        normal
5                 0          99.3        normal
6                 0          99.3        normal
7                 0          97.6        normal
8                 0          99.1        normal
9                 0          98.7        normal


In [14]:
import numpy as np
import pandas as pd

def process_activity_final(row):
    val = row['activity_value']
    lvl = row['activity_level']
    
    # Standardize existing labels to match your logic (mapping 'Sedentary' from raw data to 'rest')
    if pd.notna(lvl):
        lvl = str(lvl).lower()
        if lvl == 'sedentary':
            lvl = 'rest'
            
    # 1. Handle missing activity_value
    if pd.isna(val):
        # Look at activity_level and adjust activity_value accordingly
        if pd.notna(lvl):
            if lvl == 'rest':
                val = np.random.uniform(0.95, 1.05)
            elif lvl == 'moderate':
                val = np.random.uniform(1.2, 3.0)
            elif lvl == 'active':
                val = np.random.uniform(3.01, 5.0)
            else:
                val = np.random.uniform(1.2, 3.0) # Fallback for unrecognized strings
        # If also no activity_level, assume a random value
        else:
            rand_choice = np.random.choice(['rest', 'moderate', 'active'])
            if rand_choice == 'rest':
                val = np.random.uniform(0.95, 1.05)
            elif rand_choice == 'moderate':
                val = np.random.uniform(1.2, 3.0)
            else:
                val = np.random.uniform(3.01, 5.0)
    else:
        val = float(val)

    # 2. Adjust the activity_level strictly based on your defined value ranges
    if 0.95 <= val <= 1.05:
        final_lvl = 'rest'
    elif 1.2 <= val <= 3.0:
        final_lvl = 'moderate'
    elif val > 3.0:
        final_lvl = 'active'
    else:
        final_lvl = 'undefined' # Accounts for any raw values falling in the 1.06 - 1.19 gap
        
    row['activity_value'] = round(val, 2)
    row['activity_level'] = final_lvl
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_activity_final, axis=1)

# Verify the changes
print(df_merged[['attack_predicted', 'activity_value', 'activity_level']].head(10))

   attack_predicted  activity_value activity_level
0                 0            2.94       moderate
1                 0            1.00           rest
2                 0            2.11       moderate
3                 1            1.01           rest
4                 0            3.13         active
5                 0            2.58       moderate
6                 0            0.97           rest
7                 0            0.99           rest
8                 0            1.73       moderate
9                 0            0.97           rest


In [16]:
import numpy as np
import pandas as pd

def process_voc_overwrite(row):
    iaqi = row['IAQI']
    
    # Overwrite VOC_index entirely based on IAQI category (ignoring previous values)
    if iaqi == 'Polluted':
        voc = np.random.uniform(101.0, 300.0) # > 100
    elif iaqi == 'Excellent':
        voc = np.random.uniform(0.0, 99.9)    # < 100
    else:
        # Fallback for 'Moderate' IAQI since it was not explicitly defined
        voc = np.random.uniform(80.0, 120.0)
        
    row['VOC_index'] = round(voc, 1)
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_voc_overwrite, axis=1)

# Verify the changes
print(df_merged[['IAQI', 'VOC_index']].head(10))

        IAQI  VOC_index
0  Excellent       89.6
1  Excellent       45.5
2  Excellent        3.2
3   Polluted      190.2
4  Excellent       11.4
5  Excellent       53.8
6  Excellent        7.2
7  Excellent       82.1
8  Excellent       95.2
9  Excellent       15.7


In [19]:
import numpy as np
import pandas as pd

def process_temp_humidity(row):
    attack = row['attack_predicted']
    
    # Generate values based on clinical environmental triggers for asthma
    if attack == 1:
        # Asthma attacks are highly correlated with weather extremes.
        # We simulate a 50/50 split between "Cold & Dry" and "Hot & Humid" triggers.
        if np.random.rand() < 0.5:
            temp = np.random.uniform(0.0, 12.0)     # Cold weather (Celsius)
            humidity = np.random.uniform(15.0, 30.0) # Dry air
        else:
            temp = np.random.uniform(30.0, 42.0)    # Hot weather
            humidity = np.random.uniform(65.0, 95.0) # High humidity / Heavy air
            
    elif attack == 0:
        # Patients without an attack are more likely in the "comfort zone"
        temp = np.random.uniform(18.0, 28.0)        # Moderate room/outdoor temp
        humidity = np.random.uniform(35.0, 55.0)    # Optimal respiratory humidity
        
    else:
        # Fallback if attack_predicted is NaN
        temp = np.random.uniform(15.0, 30.0)
        humidity = np.random.uniform(30.0, 60.0)
        
    row['temp'] = round(temp, 1)
    row['humidity'] = round(humidity, 1)
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_temp_humidity, axis=1)

# Verify the changes
print(df_merged[['attack_predicted', 'temp', 'humidity']].head(10))

   attack_predicted  temp  humidity
0                 0  18.6      52.6
1                 0  24.8      43.8
2                 0  23.9      45.6
3                 1  39.2      78.8
4                 0  27.6      54.1
5                 0  19.2      45.5
6                 0  26.8      36.3
7                 0  27.6      45.5
8                 0  26.3      49.9
9                 0  21.5      46.3


In [21]:
import numpy as np
import pandas as pd

def process_heart_rate(row):
    age = row['age']
    attack = row['attack_predicted']
    activity = row['activity_level']
    spo2 = row['SpO2_category']
    
    # 1. Establish base normal HR range based on age brackets
    if age <= 6:
        min_hr, max_hr = 75, 140
    elif age <= 8:
        min_hr, max_hr = 70, 130
    elif age <= 10:
        min_hr, max_hr = 60, 130
    else: # 11-12 years old
        min_hr, max_hr = 65, 120
        
    # 2. Apply Attack (Tachycardia) and Relationship Modifiers
    if attack == 1:
        # Tachycardia: Baseline shifts completely above the normal maximum
        base_hr = np.random.uniform(max_hr + 5, max_hr + 30)
        
        # Compensatory tachycardia for low SpO2
        if spo2 == 'low':
            base_hr += np.random.uniform(10, 20)
            
        # Activity strain during an attack
        if activity == 'active':
            base_hr += np.random.uniform(10, 20)
        elif activity == 'rest':
            base_hr -= np.random.uniform(0, 10)
            
    else:
        # Normal physiology driven primarily by activity level
        if activity == 'rest':
            # Lower 40% of the normal range
            base_hr = np.random.uniform(min_hr, min_hr + (max_hr - min_hr) * 0.4)
        elif activity == 'moderate':
            # Middle of the normal range
            base_hr = np.random.uniform(min_hr + (max_hr - min_hr) * 0.3, max_hr - (max_hr - min_hr) * 0.2)
        elif activity == 'active':
            # Upper end of normal, simulating exertion
            base_hr = np.random.uniform(max_hr - 20, max_hr + 10)
        else:
            base_hr = np.random.uniform(min_hr, max_hr)
            
        # Low SpO2 in non-attack patients still elevates heart rate slightly
        if spo2 == 'low':
            base_hr += np.random.uniform(10, 20)

    # Convert to whole numbers as BPM is always an integer
    row['heart_rate'] = int(round(base_hr))
    return row

# Apply the function across all rows
df_merged = df_merged.apply(process_heart_rate, axis=1)

# Verify the relationships 
print(df_merged[['age', 'attack_predicted', 'activity_level', 'SpO2_category', 'heart_rate']].head(15))

    age  attack_predicted activity_level SpO2_category  heart_rate
0     7                 0       moderate        normal          97
1     9                 0           rest        normal          67
2     8                 0       moderate        normal         113
3     6                 1           rest           low         177
4     9                 0         active        normal         123
5     8                 0       moderate        normal         111
6    11                 0           rest        normal          78
7     8                 0           rest        normal          83
8     5                 0       moderate        normal          96
9     7                 0           rest        normal          88
10   12                 0           rest        normal          66
11    5                 0           rest        normal          88
12    9                 0         active        normal         130
13    6                 0           rest        normal        

In [22]:
print(df_merged.isna().sum())

sex                 0
age                 0
BMI                 0
IAQI                0
activity_level      0
latest_pefr         0
attack_predicted    0
IAQV                0
VOC_index           0
SpO2_percent        0
activity_value      0
temp                0
humidity            0
heart_rate          0
pefr_zone           0
SpO2_category       0
dtype: int64


In [23]:
# Count missing values in the attack_predicted column
missing_labels = df_merged['attack_predicted'].isna().sum()

print(f"Number of patients missing an attack prediction: {missing_labels}")

Number of patients missing an attack prediction: 0


In [24]:
# Save the current cleaned dataset to a CSV file (without the index column)
df_merged.to_csv('Nafas_v1.csv', index=False)

print("Cleaned file saved successfully!")

Cleaned file saved successfully!


In [25]:
class_0 = (df_merged['attack_predicted'] == 0).sum()
class_1 = (df_merged['attack_predicted'] == 1).sum()

print(f"No Attack (0): {class_0}")
print(f"Attack (1): {class_1}")

No Attack (0): 968
Attack (1): 284


In [26]:
import pandas as pd
import numpy as np

# 1. Separate the original clean data by class
df_0 = df_merged[df_merged['attack_predicted'] == 0]
df_1 = df_merged[df_merged['attack_predicted'] == 1]

# 2. Define targets for a perfect 50/50 split of 11,000 rows
target_per_class = 5500

# Calculate synthetic rows needed per class
needed_0 = target_per_class - len(df_0)  # Needs 4,532 rows
needed_1 = target_per_class - len(df_1)  # Needs 5,216 rows

# 3. Bootstrap each class independently
synth_0 = df_0.sample(n=needed_0, replace=True).reset_index(drop=True)
synth_1 = df_1.sample(n=needed_1, replace=True).reset_index(drop=True)

# Combine all synthetic rows before applying noise
df_synthetic = pd.concat([synth_0, synth_1], axis=0, ignore_index=True)

# 4. Apply Gaussian Noise to continuous columns (only on synthetic rows)
continuous_cols = ['BMI', 'latest_pefr', 'SpO2_percent', 'VOC_index', 
                   'activity_value', 'temp', 'humidity', 'heart_rate']

for col in continuous_cols:
    std_dev = df_merged[col].std()
    # Apply 2% variance
    noise = np.random.normal(0, std_dev * 0.02, size=len(df_synthetic))
    
    df_synthetic[col] = df_synthetic[col] + noise
    
    # Maintain appropriate formatting
    if col == 'heart_rate':
        df_synthetic[col] = df_synthetic[col].round().astype(int)
    elif col == 'activity_value':
        df_synthetic[col] = df_synthetic[col].round(2)
    else:
        df_synthetic[col] = df_synthetic[col].round(1)

# 5. Combine original data with the new balanced synthetic data
df_final = pd.concat([df_0, df_1, df_synthetic], axis=0, ignore_index=True)

# 6. Shuffle to mix classes and prevent sequential bias during training
df_final = df_final.sample(frac=1).reset_index(drop=True)

# Save and verify
df_final.to_csv('Nafas_11k_Balanced.csv', index=False)
print("Balanced 11,000 row dataset created successfully!")
print(df_final['attack_predicted'].value_counts())

Balanced 11,000 row dataset created successfully!
attack_predicted
1    5500
0    5500
Name: count, dtype: int64
